# Adversarial Search: Playing Dots and Boxes


## Instructions

Total Points: Undegraduates 100, graduate students 110

Complete this notebook and submit it. The notebook needs to be a complete project report with your implementation, documentation including a short discussion of how your implementation works and your design choices, and experimental results (e.g., tables and charts with simulation results) with a short discussion of what they mean. Use the provided notebook cells and insert additional code and markdown cells as needed.

## Introduction

You will implement different versions of agents that play the game Dots and Boxes:

> "Dots and Boxes is a pencil-and-paper game for two players. The game starts with an empty grid of dots. Usually two players take turns adding a single horizontal or vertical line between two unjoined adjacent dots. A player who completes the fourth side of a 1x1 box earns one point and takes another turn. A point is typically recorded by placing a mark that identifies the player in the box, such as an initial. The game ends when no more lines can be placed. The winner is the player with the most points. The board may be of any size grid." (see [Dots and Boxes on Wikipedia](https://en.wikipedia.org/wiki/Dots_and_Boxes))

You can play Dots and Boxes [here](https://www.math.ucla.edu/~tom/Games/dots&boxes.html).

## Task 1: Defining the Search Problem [10 point]

Define the components of the search problem associated with this game:

* Initial state
* Actions
* Transition model
* Test for the terminal state
* Utility for terminal states

### [Bài 10] Trả lời Task 1: các thành phần của bài toán tìm kiếm

Bàn có `rows × cols` **chấm** (mặc định của đề là 4×4 chấm, tức 3×3 ô). Hai người chơi là **+1** (đi trước) và **−1**.

| Thành phần | Định nghĩa |
| :--- | :--- |
| **Trạng thái ban đầu** | Bàn chưa có đoạn nào (`lines` rỗng), chưa ô nào có chủ (`boxes` rỗng), người +1 đến lượt. |
| **Hành động** `actions(s)` | Vẽ một đoạn ngang `('h', r, c)` hoặc dọc `('v', r, c)` chưa vẽ, nối hai chấm kề nhau. Bàn `rows × cols` chấm có $L = rows\cdot(cols-1) + (rows-1)\cdot cols$ đoạn (4×4 chấm: $L = 24$). |
| **Mô hình chuyển trạng thái** `result(s, a)` | Thêm đoạn `a` vào `lines`. Mỗi ô kề `a` (tối đa 2 ô) vừa đủ 4 cạnh thì ghi `(hàng, cột): người chơi` vào `boxes`. Nếu nước đi **đóng được ít nhất một ô thì người đó đi tiếp**, ngược lại đổi lượt. |
| **Kiểm tra kết thúc** `terminal(s)` | Đã vẽ đủ cả $L$ đoạn (lúc đó mọi ô đều đã có chủ). |
| **Utility** `utility(s, p)` | So số ô của hai người theo góc nhìn người `p`: +1 nếu nhiều ô hơn, −1 nếu ít hơn, 0 nếu bằng nhau (chỉ xảy ra khi tổng số ô chẵn). |

Khác với Tic-Tac-Toe, lượt đi **không luân phiên cố định** vì luật đi tiếp, nên trạng thái đầy đủ dùng cho tìm kiếm là
cặp *(bàn cờ, người đến lượt)*. Bàn cờ giữ đúng cấu trúc dict chung của đề (`size`, `lines`, `boxes`) để các agent
đấu được với nhau; người đến lượt do môi trường quản lý (hàm `next_player` ở Task 2).

How big is the state space? Give an estimate and explain it.

### [Bài 10] Trả lời: kích thước không gian trạng thái

Một trạng thái gồm: tập đoạn đã vẽ, chủ của từng ô đã đóng và người đến lượt.

- **Cận dưới $2^L$**: mỗi đoạn vẽ hoặc chưa vẽ, và mọi tập con của $L$ đoạn đều đạt được (vẽ theo thứ tự bất kỳ),
  nên có ít nhất $2^L$ trạng thái khác nhau.
- **Cận trên $2^L \cdot 2^B \cdot 2$** với $B$ là số ô: mỗi ô đã đóng có tối đa 2 khả năng về chủ, cộng 2 khả năng người đến lượt.
- **Bàn 4×4 chấm** ($L = 24$, $B = 9$): $2^{24} \approx 1{,}7 \cdot 10^{7} \le |S| \le 2^{34} \approx 1{,}7 \cdot 10^{10}$.

Bảng dưới tính các con số này cho nhiều kích thước: mỗi hàng hoặc cột chấm thêm vào làm $L$ tăng, nên số trạng thái tăng
theo hàm mũ (bàn chuẩn 5×5 chấm đã có ít nhất $2^{40} \approx 10^{12}$ trạng thái). Để ra quyết định, vị trí ô của ai
không quan trọng, chỉ cần **hiệu số ô** hiện tại, nên số trạng thái cần phân biệt thực tế nhỏ hơn cận trên nhiều.

In [1]:
# [Bài 10] Ước lượng kích thước không gian trạng thái và cây trò chơi theo kích thước bàn (số chấm).
import math
import pandas as pd

def count_lines(rows, cols):
    """Số đoạn thẳng có thể vẽ trên bàn rows × cols chấm."""
    return rows * (cols - 1) + (rows - 1) * cols

def count_boxes(rows, cols):
    return (rows - 1) * (cols - 1)

size_rows = []
for rows, cols in [(2, 2), (2, 3), (3, 3), (3, 4), (4, 4), (5, 5), (6, 6)]:
    L, B = count_lines(rows, cols), count_boxes(rows, cols)
    size_rows.append({
        'bàn (chấm)': f'{rows}×{cols}', 'số ô': B, 'số đoạn L': L,
        'tập đoạn đã vẽ 2^L': f'{2 ** L:.3g}',
        'cận trên trạng thái 2^L·2^B·2': f'{2 ** (L + B + 1):.3g}',
        'số ván (lá) L!': f'{math.factorial(L):.3g}',
        'số nút cây ≈ e·L!': f'{sum(math.perm(L, k) for k in range(L + 1)):.3g}',
    })
size_table = pd.DataFrame(size_rows)
size_table

,bàn (chấm),số ô,số đoạn L,tập đoạn đã vẽ 2^L,cận trên trạng thái 2^L·2^B·2,số ván (lá) L!,số nút cây ≈ e·L!
0,2×2,1,4,16,64,24,65
1,2×3,2,7,128,1.02e+03,5.04e+03,1.37e+04
2,3×3,4,12,4.1e+03,1.31e+05,4.79e+08,1.3e+09
3,3×4,6,17,1.31e+05,1.68e+07,3.56e+14,9.67e+14
4,4×4,9,24,1.68e+07,1.72e+10,6.2e+23,1.69e+24
5,5×5,16,40,1.1e+12,1.44e+17,8.16e+47,2.22e+48
6,6×6,25,60,1.15e+18,7.74e+25,8.32e+81,2.26e+82


How big is the game tree that minimax search will go through? Give an estimate and explain it.

### [Bài 10] Trả lời: kích thước cây trò chơi mà Minimax phải duyệt

- Mỗi ván **luôn kéo dài đúng $L$ nước**: mỗi nước vẽ một đoạn; luật đi tiếp chỉ đổi *ai* đi, không đổi *số* nước.
- Mọi thứ tự vẽ $L$ đoạn đều là một ván hợp lệ, nên cây Minimax đầy đủ (không cắt tỉa) có **$L!$ lá**, và tổng số nút là
  $\sum_{k=0}^{L} \frac{L!}{(L-k)!} \approx e \cdot L!$ (hai cột cuối của bảng trên).
- Bàn 4×4 chấm: $24! \approx 6{,}2 \cdot 10^{23}$ lá, khoảng $1{,}7 \cdot 10^{24}$ nút. Ngay cả bàn 3×3 chấm (chỉ 4 ô) đã có
  $12! \approx 4{,}8 \cdot 10^{8}$ lá; bàn chuẩn 5×5 chấm khoảng $10^{48}$ nút.
- Cây lớn hơn không gian trạng thái rất nhiều vì **cùng một trạng thái đạt được qua nhiều thứ tự vẽ khác nhau**
  (ví dụ bàn 4×4: $\approx 10^{24}$ nút cây so với tối đa $\approx 10^{10}$ trạng thái). Vì vậy Minimax/Alpha-Beta đầy đủ
  chỉ khả thi trên bàn rất nhỏ; với bàn lớn hơn cần cắt tỉa, sắp xếp nước đi, heuristic và cutoff (Task 3, Task 4).

## Task 2: Game Environment and Random Agent [30 point]

You need to think about a data structure to represent the board meaning he placed lines and who finished what box. There are many options. Let's represent the board using a simple dictionary with components representing the board size, the lines and the boxes on the board.

**Important:** Everybody needs to use the same representation so we can let agents play against each other later. 

In [2]:
board = {
    'size': (4, 4),  ### number of rows and columns of dots
    'lines': dict(), ### keys are the set of drawn lines
    'boxes': dict    ### keys are the boxes and the value is the player who completed each box
}

def draw_line(board, orientation, row, col):
    """
    Place a line on an exiting board.
       
    Parameters
    ----------
    board: dict
        the board
    orientation: str
        either 'h' or 'v' for horizontal or vertical
    row, col: int
        index of the starting dot for the line (starting with 0)
    
    """
    
    if orientation not in ['h', 'v']:
        return False
        
    if row < 0 or col < 0:
        return False
        
    if row >= board['size'][0] + (orientation == 'v') or col >= board['size'][1] + (orientation == 'h'):
        return False
        
    if (orientation, row, col) in board['lines']:
        return False
            
    board["lines"][(orientation, row, col)] = True
    return True
    

print(draw_line(board, "h", 1, 1))
print(draw_line(board, "v", 1, 1))

# this should not work
print(draw_line(board, "h", 1, 1))

board

True
True
False


{'size': (4, 4),
 'lines': {('h', 1, 1): True, ('v', 1, 1): True},
 'boxes': dict}

**[Bài 10] Nhận xét ô gốc của đề.** Ô trên minh họa cấu trúc dict chung (`size`, `lines`, `boxes`); nhóm giữ đúng cấu trúc này.
Có hai chỗ trong ô mẫu nhóm không dùng lại nguyên văn:
- `'boxes': dict` gán **kiểu** `dict` chứ không phải một dict rỗng, nên hàm `new_board` bên dưới tạo `'boxes': dict()`.
- `draw_line` kiểm tra biên chưa khớp với quy ước "`size` là số hàng, số cột **chấm**": trên bàn 4×4 chấm nó vẫn nhận
  `('h', 3, 3)` (chấm `(3, 4)` không tồn tại) và `('v', 3, 0)` (chấm `(4, 0)` không tồn tại). Hàm `is_valid_line` ở Task 2
  kiểm tra đúng: đoạn ngang cần $0 \le c < cols-1$, đoạn dọc cần $0 \le r < rows-1$.

Write code to display the board. **Bonus point: Post your visualization code with an example output to the discussion board. The best visualization will earn you bonus participation points in this class.**

**[Bài 10] Cách hiển thị.** `+` là chấm, `---` là đoạn ngang, `|` là đoạn dọc đã vẽ; chữ trong ô là người đã đóng ô
(**A** = người +1, **B** = người −1). Dòng cuối in số ô mỗi người và số đoạn đã vẽ. Ví dụ dưới là bàn 3×4 chấm có 6 đoạn,
người +1 đã đóng ô (0, 0).

In [3]:
# [Bài 10] Biểu diễn bàn cờ (dict như đề) và hàm hiển thị dạng chữ.
# Quy ước: 'size' = (số hàng chấm, số cột chấm); đoạn ('h', r, c) nối chấm (r, c)–(r, c+1);
# đoạn ('v', r, c) nối chấm (r, c)–(r+1, c); ô (r, c) có góc trên-trái là chấm (r, c).
# Người chơi là +1 và -1; khi hiển thị, ô của +1 ghi 'A', ô của -1 ghi 'B'.
import numpy as np

def new_board(rows=4, cols=4):
    """Bàn trống rows × cols chấm, cùng cấu trúc dict với đề."""
    return {'size': (rows, cols), 'lines': dict(), 'boxes': dict()}

def show_board(board):
    """In bàn cờ: '+' là chấm, '---' và '|' là đoạn đã vẽ, chữ trong ô là người đã đóng ô."""
    rows, cols = board['size']
    mark = {1: 'A', -1: 'B'}
    out = []
    for r in range(rows):
        dots = ''
        for c in range(cols):
            dots += '+'
            if c < cols - 1:
                dots += '---' if ('h', r, c) in board['lines'] else '   '
        out.append(dots)
        if r < rows - 1:
            sides = ''
            for c in range(cols):
                sides += '|' if ('v', r, c) in board['lines'] else ' '
                if c < cols - 1:
                    sides += f" {mark[board['boxes'][(r, c)]]} " if (r, c) in board['boxes'] else '   '
            out.append(sides)
    print('\n'.join(out))
    owners = list(board['boxes'].values())
    print(f"A (+1): {owners.count(1)} ô   B (-1): {owners.count(-1)} ô   "
          f"đã vẽ {len(board['lines'])} đoạn")

demo = new_board(3, 4)
for line in [('h', 0, 0), ('h', 1, 0), ('v', 0, 0), ('v', 0, 1), ('h', 0, 2), ('v', 1, 3)]:
    demo['lines'][line] = True
demo['boxes'][(0, 0)] = 1
show_board(demo)

+---+   +---+
| A |        
+---+   +   +
            |
+   +   +   +
A (+1): 1 ô   B (-1): 0 ô   đã vẽ 6 đoạn


Implement helper functions for:

* The transition model $result(s, a)$.
* The utility function $utility(s)$.
* Check for terminal states $terminal(s)$.
* A check for available actions in each state $actions(s)$.

__Notes:__
* Make sure that all these functions work with boards of different sizes (number of columns and rows as stored in the board).
* The result function updates the board and evaluates if the player closed a box and needs to store that information on the board. Add elements of the form `(row,col): player` to the board dictionary. `row` and `col` are the coordinates for the box and `player` is +1 or -1 representing the player. For example `(0,0): -1` means that the top-left box belongs to the other player. 
* _Important:_ Remember that a player goes again after she completes a box!

In [4]:
# [Bài 10] actions, result, terminal, utility (chạy được với mọi kích thước bàn).

def is_valid_line(board, line):
    rows, cols = board['size']
    orientation, r, c = line
    if orientation == 'h':
        return 0 <= r < rows and 0 <= c < cols - 1
    if orientation == 'v':
        return 0 <= r < rows - 1 and 0 <= c < cols
    return False

def all_lines(board):
    rows, cols = board['size']
    return ([('h', r, c) for r in range(rows) for c in range(cols - 1)] +
            [('v', r, c) for r in range(rows - 1) for c in range(cols)])

def box_sides(box):
    r, c = box
    return [('h', r, c), ('h', r + 1, c), ('v', r, c), ('v', r, c + 1)]

def adjacent_boxes(board, line):
    """Các ô (tối đa 2) có cạnh là đoạn `line`."""
    rows, cols = board['size']
    orientation, r, c = line
    candidates = [(r - 1, c), (r, c)] if orientation == 'h' else [(r, c - 1), (r, c)]
    return [(br, bc) for br, bc in candidates if 0 <= br < rows - 1 and 0 <= bc < cols - 1]

def actions(board):
    """Các đoạn còn vẽ được."""
    return [line for line in all_lines(board) if line not in board['lines']]

def result(board, player, action):
    """Bàn mới sau khi `player` (+1/-1) vẽ đoạn `action`; không sửa bàn cũ.
    Ô nào vừa đủ 4 cạnh thì ghi (row, col): player vào board['boxes']."""
    if not is_valid_line(board, action) or action in board['lines']:
        raise ValueError(f'Nước đi không hợp lệ {action} của người chơi {player}')
    new = {'size': board['size'], 'lines': dict(board['lines']), 'boxes': dict(board['boxes'])}
    new['lines'][action] = True
    for box in adjacent_boxes(new, action):
        if all(side in new['lines'] for side in box_sides(box)):
            new['boxes'][box] = player
    return new

def closed_box(board, new_board):
    """Nước vừa đi có đóng được ô nào không (đóng ô thì được đi tiếp)."""
    return len(new_board['boxes']) > len(board['boxes'])

def next_player(board, new_board, player):
    """Người đi tiếp: vẫn là `player` nếu vừa đóng ô, ngược lại đổi lượt."""
    return player if closed_box(board, new_board) else -player

def terminal(board):
    """Hết đoạn để vẽ thì ván kết thúc."""
    rows, cols = board['size']
    return len(board['lines']) == rows * (cols - 1) + (rows - 1) * cols

def score(board, player=1):
    """Số ô của `player` trừ số ô của đối thủ."""
    owners = list(board['boxes'].values())
    return owners.count(player) - owners.count(-player)

def utility(board, player=1):
    """+1 thắng, -1 thua, 0 hòa theo góc nhìn `player`; None nếu ván chưa kết thúc."""
    if not terminal(board):
        return None
    return int(np.sign(score(board, player)))

In [5]:
# [Bài 10] Kiểm tra các hàm trên những tình huống cần thiết.
b = new_board(3, 3)                       # 3×3 chấm = 4 ô, 12 đoạn
print('Bàn trống:', len(actions(b)), 'đoạn vẽ được; kết thúc?', terminal(b))
assert len(actions(b)) == 12 and not terminal(b) and utility(b) is None

# 1) Vẽ 3 cạnh của ô (0, 0), người -1 vẽ cạnh thứ 4 thì đóng ô và được đi tiếp.
for line in [('h', 0, 0), ('v', 0, 0), ('h', 1, 0)]:
    b = result(b, 1, line)
after = result(b, -1, ('v', 0, 1))
print('\n1) Người -1 vẽ cạnh thứ 4 của ô (0, 0):')
show_board(after)
print('   ô (0, 0) thuộc', after['boxes'][(0, 0)], '; người đi tiếp:', next_player(b, after, -1))
assert after['boxes'] == {(0, 0): -1} and next_player(b, after, -1) == -1
assert ('v', 0, 1) not in b['lines']      # result không sửa bàn cũ

# 2) Nước không đóng ô thì đổi lượt.
after2 = result(b, -1, ('h', 0, 1))
assert after2['boxes'] == {} and next_player(b, after2, -1) == 1
print('\n2) Nước không đóng ô: người đi tiếp là', next_player(b, after2, -1))

# 3) Một đoạn đóng cùng lúc 2 ô (đoạn chung của ô (0, 0) và ô (0, 1)).
b3 = new_board(2, 3)
for line in [('h', 0, 0), ('h', 1, 0), ('v', 0, 0), ('h', 0, 1), ('h', 1, 1), ('v', 0, 2)]:
    b3 = result(b3, 1, line)
b3_after = result(b3, -1, ('v', 0, 1))
print('\n3) Vẽ đoạn chung đóng 2 ô cùng lúc:')
show_board(b3_after)
assert b3_after['boxes'] == {(0, 0): -1, (0, 1): -1} and terminal(b3_after)
print('   utility theo người -1:', utility(b3_after, -1), '; theo người +1:', utility(b3_after, 1))
assert utility(b3_after, -1) == 1 and utility(b3_after, 1) == -1

# 4) Nước không hợp lệ bị từ chối: vẽ lại đoạn đã có, đoạn ra ngoài bàn.
for bad in [('h', 0, 0), ('h', 0, 2), ('v', 2, 0), ('x', 0, 0)]:
    try:
        result(b, 1, bad); print('LỖI: chấp nhận', bad)
    except ValueError:
        print('4) Từ chối nước không hợp lệ', bad)

# 5) Bàn khác kích thước: số đoạn đúng công thức rows·(cols−1) + (rows−1)·cols.
for rows, cols in [(2, 2), (3, 5), (5, 5)]:
    assert len(actions(new_board(rows, cols))) == count_lines(rows, cols)
print('5) Số nước đi ở bàn 2×2, 3×5, 5×5:', [len(actions(new_board(r, c))) for r, c in [(2, 2), (3, 5), (5, 5)]])

Bàn trống: 12 đoạn vẽ được; kết thúc? False

1) Người -1 vẽ cạnh thứ 4 của ô (0, 0):
+---+   +
| B |    
+---+   +
         
+   +   +
A (+1): 0 ô   B (-1): 1 ô   đã vẽ 4 đoạn
   ô (0, 0) thuộc -1 ; người đi tiếp: -1

2) Nước không đóng ô: người đi tiếp là 1

3) Vẽ đoạn chung đóng 2 ô cùng lúc:
+---+---+
| B | B |
+---+---+
A (+1): 0 ô   B (-1): 2 ô   đã vẽ 7 đoạn
   utility theo người -1: 1 ; theo người +1: -1
4) Từ chối nước không hợp lệ ('h', 0, 0)
4) Từ chối nước không hợp lệ ('h', 0, 2)
4) Từ chối nước không hợp lệ ('v', 2, 0)
4) Từ chối nước không hợp lệ ('x', 0, 0)
5) Số nước đi ở bàn 2×2, 3×5, 5×5: [4, 22, 40]


**[Bài 10] Nhận xét các hàm của Task 2.**
- Tất cả các hàm chỉ đọc `board['size']`, nên **chạy được với mọi kích thước bàn** (kiểm tra 5 ở trên: 4, 22 và 40 nước đi
  với bàn 2×2, 3×5 và 5×5 chấm, đúng công thức $L$).
- `result(board, player, action)` có cùng thứ tự tham số với `tictactoe.py` và **trả về bàn mới, không sửa bàn cũ**,
  để Alpha-Beta ở Task 3 thử nhiều nước trên cùng một bàn mà không phải hoàn tác.
- Sau mỗi nước chỉ cần xét **tối đa 2 ô kề đoạn vừa vẽ** (`adjacent_boxes`), không phải quét cả bàn.
- **Luật đi tiếp** nằm ở `next_player`: so số ô trước và sau nước đi, có ô mới thì người đó đi tiếp. Tách riêng hàm này vì
  cấu trúc dict chung của đề không có khóa "người đến lượt".
- Các kiểm tra ở trên cover đủ các tình huống quan trọng: đóng ô được đi tiếp (1), không đóng ô thì đổi lượt (2), một
  đoạn đóng **cùng lúc 2 ô** (3), từ chối nước vẽ lại và nước ra ngoài bàn (4).

Implement an agent that plays randomly. Make sure the agent function receives as the percept the board and returns a valid action. Use an agent function definition with the following signature (arguments):

`def random_player(board, player = None): ...`

The argument `player` is used for agents that do not store what side they are playing. The value passed on by the environment should be 1 ot -1 for playerred and yellow, respectively.  See [Experiments section for tic-tac-toe](https://nbviewer.org/github/mhahsler/CS7320-AI/blob/master/Games/tictactoe_and_or_tree_search.ipynb#Experiments) for an example.

In [6]:
# [Bài 10] Random agent: nhận bàn cờ (percept), trả một nước hợp lệ.
def random_player(board, player=None):
    """Chọn ngẫu nhiên đều một đoạn còn trống; `player` không dùng."""
    moves = actions(board)
    return moves[np.random.randint(len(moves))]

# Môi trường: hai agent lần lượt đi, đóng ô thì được đi tiếp.
def play_game(agent1, agent2, size=(4, 4), show_final_board=False):
    """Chơi 1 ván, agent1 là người +1 và đi trước. Trả utility theo góc nhìn +1."""
    board, player = new_board(*size), 1
    agents = {1: agent1, -1: agent2}
    while not terminal(board):
        action = agents[player](board, player)
        new = result(board, player, action)
        player = next_player(board, new, player)
        board = new
    if show_final_board:
        show_board(board)
    return utility(board, 1)

def play(agent1, agent2, N=1000, size=(4, 4)):
    """Cho hai agent đấu N ván; agent1 (+1) luôn đi trước."""
    results = {1: 0, -1: 0, 0: 0}
    for _ in range(N):
        results[play_game(agent1, agent2, size)] += 1
    return results

np.random.seed(42)
print('Một ván mẫu trên bàn 4×4 chấm, kết quả theo người +1:',
      play_game(random_player, random_player, show_final_board=True))

+---+---+---+
| A | A | A |
+---+---+---+
| B | B | A |
+---+---+---+
| B | B | A |
+---+---+---+
A (+1): 5 ô   B (-1): 4 ô   đã vẽ 24 đoạn
Một ván mẫu trên bàn 4×4 chấm, kết quả theo người +1: 1


Let two random agents play against each other 1000 times. Look at the [Experiments section for tic-tac-toe](https://nbviewer.org/github/mhahsler/CS7320-AI/blob/master/Games/tictactoe_and_or_tree_search.ipynb#Experiments) to see how the environment uses the agent functions to play against each other.

How often does each player win? Is the result expected?

In [7]:
# [Bài 10] Hai random agent đấu 1000 ván trên nhiều kích thước bàn (seed 42).
import time
rows_out = []
for size in [(3, 3), (4, 4), (5, 5)]:
    np.random.seed(42)
    start = time.perf_counter()
    res = play(random_player, random_player, N=1000, size=size)
    elapsed = time.perf_counter() - start
    rows_out.append({'bàn (chấm)': f'{size[0]}×{size[1]}', 'số ô': count_boxes(*size),
                     'người +1 thắng (đi trước)': res[1], 'người -1 thắng': res[-1], 'hòa': res[0],
                     'tỷ lệ +1 thắng': res[1] / 1000, 'thời gian (s)': round(elapsed, 2)})
random_table = pd.DataFrame(rows_out)
random_table

,bàn (chấm),số ô,người +1 thắng (đi trước),người -1 thắng,hòa,tỷ lệ +1 thắng,thời gian (s)
0,3×3,4,400,432,168,0.400,0.08
1,4×4,9,490,510,0,0.490,0.17
2,5×5,16,451,466,83,0.451,0.34


**[Bài 10] Nhận xét: hai random agent đấu 1000 ván (người +1 luôn đi trước, seed 42).**
- **Bàn 4×4 chấm (mặc định của đề, 9 ô)**: người đi trước thắng **490**, người đi sau thắng **510**, **không có hòa**
  vì tổng số ô lẻ nên luôn có người hơn.
- Bàn 3×3 chấm (4 ô): **400 / 432 / 168 hòa**; bàn 5×5 chấm (16 ô): **451 / 466 / 83 hòa**. Hòa chỉ xảy ra ở bàn có số ô chẵn.
- **Kết quả này đúng như dự đoán.** Random agent không có chiến lược nên không khai thác được lợi thế nào; hai bên gần như
  chia đôi. Ở bàn 4×4, người đi trước thắng 490 ván, chỉ lệch 10 ván so với mức chia đều 500, trong khi độ lệch chuẩn của
  số ván thắng là khoảng $\sqrt{1000 \cdot 0{,}5 \cdot 0{,}5} \approx 16$ ván (khoảng 95%: 500 ± 31). Hai bàn còn lại cũng chỉ
  chênh 32 và 15 ván trên 1000. Vì vậy **không có lợi thế rõ ràng cho người đi trước** khi cả hai chơi ngẫu nhiên;
  chênh lệch quan sát được là dao động ngẫu nhiên.
- Thời gian 1000 ván chỉ dưới một giây ở cả ba kích thước, nên môi trường đủ nhanh cho các thí nghiệm ở Ngày 2 và 3.

## Task 3: Minimax Search with Alpha-Beta Pruning [30 points]

### Implement the search starting.

Implement the search starting from a given board and specifying the player and put it into an agent function.
You can use code from the [tic-tac-toe example](https://nbviewer.org/github/mhahsler/CS7320-AI/blob/master/Games/tictactoe_alpha_beta_tree_search.ipynb).

__Notes:__ 
* Make sure that all your agent functions have a signature consistent with the random agent above.
* The search space for larger board may be too large. You can experiment with smaller boards.
* Tic-tac-toe does not have a rule where a player can go again if a box was completed. You need to adapt the tree search to reflect that rule.

In [8]:
# Your code/ answer goes here.

Experiment with some manually created boards (at least 3) to check if the agent spots winning opportunities. Discuss the results.

In [9]:
# Your code/ answer goes here.

How long does it take to make a move? Start with a smaller board make the board larger. What is the largest board you can solve?

In [10]:
# Your code/ answer goes here.

### Move ordering

Starting the search with better moves will increase the efficiency of alpha-beta pruning. Describe and implement a simple move ordering strategy. 

Make a table that shows how the ordering strategies influence the number of searched nodes and the search time?

In [11]:
# Your code/ answer goes here.

### The first few moves

Start with an empty board. This is the worst case scenario for minimax search with alpha-beta pruning since it needs solve all possible games that can be played (minus some pruning) before making the decision. What can you do? 

In [12]:
# Your code/ answer goes here.

### Playtime

Let the Minimax Search agent play a random agent on a small board. Analyze wins, losses and draws.

In [13]:
# Your code/ answer goes here.

## Task 4: Heuristic Alpha-Beta Tree Search [30 points] 

### Heuristic evaluation function

Define and implement a heuristic evaluation function.

In [14]:
# Your code/ answer goes here.

### Cutting off search 

Modify your Minimax Search with Alpha-Beta Pruning to cut off search at a specified depth and use the heuristic evaluation function. Experiment with different cutoff values.

In [15]:
# Your code/ answer goes here.

How many nodes are searched and how long does it take to make a move? Start with a smaller board with 4 columns and make the board larger by adding columns.

In [16]:
# Your code/ answer goes here.

### Playtime

Let two heuristic search agents (different cutoff depth, different heuristic evaluation function) compete against each other on a reasonably sized board. Since there is no randomness, you only need to let them play once.

In [17]:
# Your code/ answer goes here.

## Tournament task [+1 to 5% bonus on your course grade; will be assigned separately]

Find another student and let your best agent play against the other student's best player. You are allowed to use any improvements you like as long as you code it yourself. We will set up a class tournament on Canvas. This tournament will continue after the submission deadline.

## Graduate student advanced task: Pure Monte Carlo Search and Best First Move [10 point]

__Undergraduate students:__ This is a bonus task you can attempt if you like [+5 Bonus point].

### Pure Monte Carlo Search

Implement Pure Monte Carlo Search (see [tic-tac-toe-example](tictactoe_pure_monte_carlo_search.ipynb)) and investigate how this search performs on the test boards that you have used above. 

In [18]:
# Your code/ answer goes here.

### Best First Move

How would you determine what the best first move for a standard board ($5 \times 5$) is? You can use Pure Monte Carlo Search or any algorithms that you have implemented above.

In [19]:
# Your code/ answer goes here.